# 01 — Exploratory Data Analysis
**Month 1, Week 2 Deliverable**

Goals:
1. Scan both datasets and build the manifest CSV
2. Understand class distribution per hive
3. Visualise waveforms and mel-spectrograms
4. Find the actual hive IDs → update `HIVE_SPLITS` in `config.py`


In [ ]:
import sys, os
sys.path.insert(0, '..')  # so we can import config, data, models from root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import config
from data import build_manifest, load_and_resample, segment_audio, to_mel_spectrogram

print('Config loaded. Sample rate:', config.SAMPLE_RATE)


## Step 1 — Build Manifest

In [ ]:
# Only run this once after downloading both datasets
# It scans data/raw/tbon/ and data/raw/queen_noqueen/ and creates manifest.csv

manifest = build_manifest(
    tbon_dir  = config.DATA_RAW_TBON,
    queen_dir = config.DATA_RAW_QUEEN,
)
manifest.head(10)


## Step 2 — Class Distribution Per Hive

In [ ]:
manifest = pd.read_csv(config.MANIFEST_CSV)
print('Total files:', len(manifest))
print('\nAll unique hive IDs found:')
print(manifest['hive_id'].unique())
print('\n⚠ Copy these hive IDs into HIVE_SPLITS in config.py!')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per-label count
manifest['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#22c55e', '#ef4444'])
axes[0].set_title('Overall Class Distribution')
axes[0].set_ylabel('Number of Files')
axes[0].tick_params(axis='x', rotation=0)

# Per-hive, per-label count
hive_label = manifest.groupby(['hive_id', 'label']).size().unstack(fill_value=0)
hive_label.plot(kind='bar', ax=axes[1], color=['#22c55e', '#ef4444'])
axes[1].set_title('Class Distribution per Hive')
axes[1].set_ylabel('Number of Files')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print('Note any imbalance — we handle it with class-weighted loss in train.py')


## Step 3 — Waveform Visualisation

In [ ]:
# Pick one queen_present and one queen_absent file to visualise
sample_present = manifest[manifest['label'] == 'queen_present'].iloc[0]['filepath']
sample_absent  = manifest[manifest['label'] == 'queen_absent'].iloc[0]['filepath']

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

for ax, fpath, title, color in [
    (axes[0], sample_present, 'Queen Present',  '#22c55e'),
    (axes[1], sample_absent,  'Queen Absent',   '#ef4444'),
]:
    wav, sr = load_and_resample(fpath)
    t = np.linspace(0, len(wav)/sr, len(wav))
    ax.plot(t, wav, color=color, linewidth=0.5, alpha=0.8)
    ax.set_title(f'{title} — {os.path.basename(fpath)}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()


## Step 4 — Mel-Spectrogram Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, fpath, title in [
    (axes[0], sample_present, 'Queen Present'),
    (axes[1], sample_absent,  'Queen Absent'),
]:
    wav, sr = load_and_resample(fpath)
    segs    = segment_audio(wav, sr)
    if segs:
        mel = librosa.feature.melspectrogram(
            y=segs[0], sr=sr, n_fft=config.N_FFT,
            hop_length=config.HOP_LENGTH, n_mels=config.N_MELS, fmax=config.FMAX
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(
            mel_db, sr=sr, hop_length=config.HOP_LENGTH,
            x_axis='time', y_axis='mel', fmax=config.FMAX, ax=ax, cmap='magma'
        )
        ax.set_title(title)
        fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.suptitle('Mel-Spectrograms — this is what the CNN sees', fontsize=13)
plt.tight_layout()
plt.show()


## Step 5 — Update config.py

After running Step 2 above, copy the actual hive IDs into `config.py` → `HIVE_SPLITS`:

```python
HIVE_SPLITS = {
    'train': ['hive_id_1', 'hive_id_2', ...],
    'val':   ['hive_id_N'],
    'test':  ['hive_id_M', ...],  # held-out unseen hives
}
```

Then re-run: `python data.py --build`


In [ ]:
print('Duration statistics:')
print(manifest['duration_sec'].describe().round(2))
manifest['duration_sec'].hist(bins=40, color='#f59e0b', edgecolor='black')
plt.title('Audio file duration distribution')
plt.xlabel('Duration (seconds)')
plt.ylabel('Count')
plt.show()
